# 🎮 Minecraft Server Manager
### Powered by PaperMC

All server files are stored locally in the `MinecraftServers/` folder next to this notebook.

| Step | Cell | What it does |
|------|------|--------------|
| 1 | **Cell 1** | Set up the environment & servers directory |
| 2 | **Cell 2** | Create a new server or load an existing one |
| 3 | **Cell 3** | Start the server & open the console |
| 4 | **Cell 4** _(optional)_ | Edit server settings (server.properties) |

> **Tip:** Run the cells from top to bottom in order. Your world is saved in `MinecraftServers/<name>/world/`.
> **Note:** The Minecraft server listens on port 25565. Use an ngrok/playit.gg tunnel to connect from outside the notebook environment.


In [ ]:
# ============================================================
# CELL 1 — Set Up Environment
# ============================================================

import os
import re
import json
import html as _html_mod
import hashlib
import shutil
import subprocess
import threading
import tempfile
import time
import atexit
import queue
import platform
import requests
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ── Global state shared across all cells ─────────────────────
# Preserve state across Cell 1 re-runs so a running server's config
# is not lost.  Only initialise if state does not yet exist.
if 'state' not in globals():
    state = {
        'server_path': None,
        'server_name': None,
        'java_cmd':    None,
        'jar_path':    None,
        'java_ver':    None,
    }

# ── Local servers directory ───────────────────────────────
# Servers are stored locally in ./MinecraftServers (relative to this notebook)
SERVERS_DIR = Path('./MinecraftServers')
SERVERS_DIR.mkdir(parents=True, exist_ok=True)

# ── CSS for monospace widgets ─────────────────────────────
# .mc-console → dark terminal console  .mc-editor → server.properties editor
display(HTML("""
<style>
  .mc-console textarea {
    font-family: 'Courier New', Courier, monospace !important;
    font-size: 12px !important;
    background: #1e1e1e !important;
    color: #d4d4d4 !important;
    border: 1px solid #444 !important;
  }
  .mc-console textarea:disabled { opacity: 1 !important; }
  .mc-editor textarea {
    font-family: 'Courier New', Courier, monospace !important;
    font-size: 12px !important;
  }
</style>
"""))

print('✅ Environment ready!')
print(f'📁 Servers directory: {SERVERS_DIR.resolve()}')

# List existing servers — skip hidden directories (e.g. .Trash-1000)
existing = sorted(
    d.name for d in SERVERS_DIR.iterdir()
    if d.is_dir() and not d.name.startswith('.')
)
if existing:
    print(f'\n📋 Found {len(existing)} existing server(s):')
    for name in existing:
        meta_file = SERVERS_DIR / name / '.mc_meta.json'
        if meta_file.exists():
            try:
                meta = json.loads(meta_file.read_text(encoding='utf-8'))
                print(f'   • {name}  (MC {meta.get("mc_version", "?")})')
            except Exception:
                print(f'   • {name}')
        else:
            print(f'   • {name}')
else:
    print('\n📭 No existing servers found — create one in Cell 2.')

print('\n▶ Run Cell 2 next.')


In [ ]:
# ============================================================
# CELL 2 — Choose or Create a Server
# ============================================================

if 'SERVERS_DIR' not in globals():
    raise RuntimeError('\u274c Please run Cell 1 first before running Cell 2.')

# ── PaperMC API helpers ───────────────────────────────────
PAPER_API = 'https://api.papermc.io/v2'

def _get_with_retry(url, retries=3, **kwargs):
    """GET with automatic retry on transient network errors (5xx / connection)."""
    for attempt in range(retries):
        try:
            r = requests.get(url, **kwargs)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            # Don't retry client errors (4xx) — they won't succeed on retry
            if hasattr(e, 'response') and e.response is not None and e.response.status_code < 500:
                raise
            if attempt == retries - 1:
                raise
            print(f'   ⚠️  Request failed ({e}), retrying ({attempt+2}/{retries})...')
            time.sleep(2 ** attempt)

def get_paper_versions():
    """Fetch all available Minecraft versions from PaperMC (newest first)."""
    r = _get_with_retry(f'{PAPER_API}/projects/paper', timeout=30)
    return list(reversed(r.json().get('versions', [])))

def get_latest_build(mc_version):
    """Get the latest stable PaperMC build info for a Minecraft version."""
    r = _get_with_retry(
        f'{PAPER_API}/projects/paper/versions/{mc_version}/builds',
        timeout=30
    )
    builds = r.json().get('builds', [])
    stable = [b for b in builds if b.get('channel') == 'default']
    target = stable if stable else builds
    return target[-1] if target else None

def download_paper_jar(mc_version, build_info, dest_path, progress_widget=None):
    """
    Download PaperMC JAR to a temp file first, then move to dest_path.
    Verifies SHA-256 checksum when provided by PaperMC API.
    Hides the progress bar whether download succeeds or fails.
    """
    try:
        build_num = build_info['build']
        filename  = build_info['downloads']['application']['name']
    except (KeyError, TypeError) as e:
        raise ValueError(f'Unexpected PaperMC API response format: {e}') from e
    url = (
        f'{PAPER_API}/projects/paper/versions/{mc_version}'
        f'/builds/{build_num}/downloads/{filename}'
    )
    print(f'⬇️  Downloading PaperMC {mc_version} build {build_num}...')

    r = requests.get(url, stream=True, timeout=(30, 300))  # (connect, read)
    r.raise_for_status()

    total = int(r.headers.get('content-length', 0))
    if progress_widget and total:
        progress_widget.max   = total
        progress_widget.value = 0
        progress_widget.layout.display = 'flex'

    # Use a proper temp file to avoid collisions between concurrent downloads
    tmp_fd, tmp_path_str = tempfile.mkstemp(prefix='paper_dl_', suffix='.tmp', dir=tempfile.gettempdir())
    tmp_path = Path(tmp_path_str)

    try:
        downloaded = 0
        sha256_ctx = hashlib.sha256()
        with os.fdopen(tmp_fd, 'wb') as f:
            for chunk in r.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
                    sha256_ctx.update(chunk)
                    downloaded += len(chunk)
                    if progress_widget and total:
                        progress_widget.value = downloaded
        # shutil.move handles cross-filesystem copies correctly
        shutil.move(str(tmp_path), str(dest_path))
        if progress_widget:
            progress_widget.layout.display = 'none'
        print(f'✅ Downloaded {downloaded / 1024 / 1024:.1f} MB')
        # Verify integrity using the hash computed inline — no need to re-read the file
        expected_sha256 = build_info['downloads']['application'].get('sha256')
        if expected_sha256:
            if sha256_ctx.hexdigest() != expected_sha256:
                dest_path.unlink()
                raise ValueError(
                    f'SHA256 mismatch — download is corrupt.\n'
                    f'Expected: {expected_sha256}\n'
                    f'Got:      {sha256_ctx.hexdigest()}'
                )
            print('✅ Checksum verified.')
    except Exception:
        if tmp_path.exists():
            tmp_path.unlink()
        if progress_widget:
            progress_widget.layout.display = 'none'
        r.close()  # release the connection back to the pool
        raise

# ── Java version requirements ──────────────────────────────
# Official Minecraft → Java requirements:
#   1.21+         → Java 21
#   1.20.5–1.20.6 → Java 21   (patch-level check)
#   1.17–1.20.4   → Java 17
#   < 1.17        → Java 11   (safer than Java 8; runs all older versions)
def java_version_for_mc(mc_version_str):
    """Return required Java major version for a Minecraft version string."""
    parts = mc_version_str.strip().split('.')
    try:
        # Strip non-numeric suffixes (e.g. '21-rc1' -> '21')
        minor = int(re.sub(r'[^0-9].*', '', parts[1])) if len(parts) > 1 else 0
    except (ValueError, IndexError):
        minor = 0
    try:
        patch = int(re.sub(r'[^0-9].*', '', parts[2])) if len(parts) > 2 else 0
    except (ValueError, IndexError):
        patch = 0
    if minor >= 21:
        return 21
    if minor == 20 and patch >= 5:   # 1.20.5+ needs Java 21
        return 21
    if minor >= 17:
        return 17
    return 11

def _detect_environment():
    """Detect whether we are running on Google Colab, Replit, or generic Linux."""
    if os.path.exists('/content') and 'COLAB_RELEASE_TAG' in os.environ:
        return 'colab'
    if 'REPL_ID' in os.environ or os.path.exists('/home/runner'):
        return 'replit'
    return 'generic'

def install_java(required_version):
    """Ensure the correct Java version is installed; return its binary path."""
    print(f'☕ Checking for Java {required_version}...')

    try:
        result      = subprocess.run(['java', '-version'], capture_output=True, text=True, timeout=15)
        version_out = result.stderr or result.stdout
    except (FileNotFoundError, subprocess.TimeoutExpired):
        version_out = ''
    match = re.search(r'version "(\d+)', version_out)
    if match:
        installed_major = int(match.group(1))
        if installed_major == 1:   # old-style "1.8.0_xxx"
            m2 = re.search(r'version "1\.(\d+)', version_out)
            installed_major = int(m2.group(1)) if m2 else 8
        if installed_major >= required_version:
            print(
                f'   ✅ Java {installed_major} already installed '
                f'— meets Java {required_version} requirement.'
            )
            return _find_java_binary(required_version)

    # Java not found or too old — try to install it
    env = _detect_environment()
    print(f'   ⚠️  Java {required_version} not found on PATH — attempting install...')

    if env in ('colab', 'generic'):
        # Colab and most Linux environments use apt-get
        _apt_install_java(required_version)
    # On Replit/Nix, Java should be pre-installed via Nix; skip apt-get.

    java_bin = _find_java_binary(required_version)
    try:
        verify  = subprocess.run([java_bin, '-version'], capture_output=True, text=True, timeout=15)
        ver_str = (verify.stderr or verify.stdout).split('\n')[0].strip()
        print(f'   ✅ Java ready: {ver_str}')
    except (FileNotFoundError, subprocess.TimeoutExpired):
        raise RuntimeError(
            f'Java {required_version} could not be found or installed. '
            f'Please install Java {required_version}+ manually and re-run this cell.'
        )
    return java_bin

def _apt_install_java(required_version):
    """Install Java via apt-get (works on Colab, Ubuntu, Debian)."""
    # Map required version to the closest available OpenJDK package
    # apt usually has 11, 17, 21 depending on the distro release
    pkg = f'openjdk-{required_version}-jre-headless'
    print(f'   📦 Installing {pkg} via apt-get (this may take a minute)...')
    try:
        subprocess.run(
            ['apt-get', 'update', '-qq'],
            capture_output=True, text=True, timeout=120
        )
        result = subprocess.run(
            ['apt-get', 'install', '-y', '-qq', pkg],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode != 0:
            # Try fallback: install the highest available version
            for fallback_ver in [21, 17, 11]:
                if fallback_ver >= required_version:
                    fb_pkg = f'openjdk-{fallback_ver}-jre-headless'
                    if fb_pkg == pkg:
                        continue
                    print(f'   ⚠️  {pkg} not available, trying {fb_pkg}...')
                    fb_result = subprocess.run(
                        ['apt-get', 'install', '-y', '-qq', fb_pkg],
                        capture_output=True, text=True, timeout=300
                    )
                    if fb_result.returncode == 0:
                        print(f'   ✅ Installed {fb_pkg}.')
                        return
            print(f'   ⚠️  apt-get install failed: {result.stderr.strip()}')
        else:
            print(f'   ✅ Installed {pkg}.')
    except FileNotFoundError:
        print('   ⚠️  apt-get not available on this system.')
    except subprocess.TimeoutExpired:
        print('   ⚠️  apt-get timed out. Check your internet connection.')

def _find_java_binary(required_version):
    """Return best-matching java binary; prefers version-specific paths, then PATH."""
    # 1. Debian/Ubuntu version-specific paths first — ensures we pick the
    #    correct version even when an older Java sits on PATH.
    # Detect architecture dynamically for non-x86/arm64 systems (riscv64, ppc64el, etc.)
    arch = platform.machine()   # e.g. 'x86_64', 'aarch64', 'riscv64'
    _arch_map = {'x86_64': 'amd64', 'aarch64': 'arm64', 'armv7l': 'armhf'}
    deb_arch = _arch_map.get(arch, arch)
    seen, order = set(), []
    for v in [required_version, 21, 17, 11]:
        if v not in seen and v >= required_version:
            seen.add(v)
            order.append(v)
    for v in order:
        for path in [
            f'/usr/lib/jvm/java-{v}-openjdk-{deb_arch}/bin/java',
            f'/usr/lib/jvm/java-{v}-openjdk/bin/java',
        ]:
            if os.path.exists(path):
                return path
    # 2. Fall back to PATH — covers Nix (Replit), Docker, and other environments
    java_on_path = shutil.which('java')
    if java_on_path:
        return java_on_path
    return 'java'   # last-resort fallback

# ── JVM flags (Aikar's recommended + Java 17+ module opens) ──
JVM_FLAGS_COMMON = [
    '-XX:+UseG1GC',
    '-XX:MaxGCPauseMillis=200',
    '-XX:+DisableExplicitGC',
    '-XX:G1NewSizePercent=30',
    '-XX:G1MaxNewSizePercent=40',
    '-XX:G1HeapRegionSize=8M',
    '-XX:G1ReservePercent=20',
    '-XX:G1HeapWastePercent=5',
    '-XX:G1MixedGCCountTarget=4',
    '-XX:InitiatingHeapOccupancyPercent=15',
    '-XX:G1MixedGCLiveThresholdPercent=90',
    '-XX:G1RSetUpdatingPauseTimePercent=5',
    '-XX:SurvivorRatio=32',
    '-XX:+PerfDisableSharedMem',
    '-XX:MaxTenuringThreshold=1',
    '-Dusing.aikars.flags=https://mcflags.emc.gs',
    '-Daikars.new.flags=true',
    '-Dfile.encoding=UTF-8',   # ensures plugins read config files correctly
]
JVM_FLAGS_MODERN = [
    '--add-opens=java.base/java.lang=ALL-UNNAMED',
    '--add-opens=java.base/java.io=ALL-UNNAMED',
    '--add-opens=java.base/java.util=ALL-UNNAMED',
    '--add-opens=java.base/java.util.concurrent=ALL-UNNAMED',
    '--add-opens=java.base/java.net=ALL-UNNAMED',
]

def _detect_ram():
    """Return (xms_mb, xmx_mb) based on available system memory."""
    try:
        total_mb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') // (1024 * 1024)
        xmx_gb = max(1, min(int(total_mb * 0.75 / 1024), 8))  # 75% of RAM, max 8 GB
        xms_mb = min(512, xmx_gb * 1024 // 4)  # 25% of max, at most 512 MB
        return xms_mb, xmx_gb * 1024
    except Exception:
        return 512, 2048  # safe fallback

def build_jvm_flags(java_version):
    """Return the full JVM flag list for the given Java major version."""
    xms_mb, xmx_mb = _detect_ram()
    flags = [f'-Xmx{xmx_mb}M', f'-Xms{xms_mb}M'] + list(JVM_FLAGS_COMMON)
    # These flags were deprecated/removed in Java 21
    if java_version < 21:
        flags.append('-XX:+UnlockExperimentalVMOptions')
        flags.append('-XX:+ParallelRefProcEnabled')
    if java_version >= 17:
        flags = flags + JVM_FLAGS_MODERN
    return flags

# ── UI state ──────────────────────────────────────────
versions_fetched = {'done': False, 'name': None}

# ── Build widget list (skip hidden dirs, sorted) ──────────────
existing_servers = sorted(
    d.name for d in SERVERS_DIR.iterdir()
    if d.is_dir() and not d.name.startswith('.')
)
choices = ['➕ Create a new server'] + existing_servers

choice_select = widgets.RadioButtons(
    options=choices,
    description='Server:',
    layout={'width': 'max-content'},
    style={'description_width': 'initial'},
)

name_input = widgets.Text(
    placeholder='e.g. MySurvivalServer',
    description='Server name:',
    layout={'width': '420px'},
    style={'description_width': 'initial'},
)

name_box = widgets.VBox([
    widgets.HTML('<b>📝 Enter a name for your new server:</b>'),
    name_input,
])

version_select = widgets.Dropdown(
    description='MC version:',
    style={'description_width': 'initial'},
    layout={'width': '320px'},
)

java_info_label = widgets.HTML('')

version_box = widgets.VBox([
    widgets.HTML('<b>🔖 Select Minecraft version (PaperMC — newest first):</b>'),
    version_select,
    java_info_label,
])
version_box.layout.display = 'none'

progress_bar = widgets.IntProgress(
    min=0, max=100, value=0,
    description='Downloading:',
    bar_style='info',
    layout=widgets.Layout(width='60%', display='none'),
    style={'description_width': 'initial'},
)

setup_btn = widgets.Button(
    description='Fetch Versions',
    button_style='info',
    icon='search',
    layout={'width': '200px', 'margin': '10px 0'},
)

out = widgets.Output()

def _update_java_info(change=None):
    if version_select.value:
        jv = java_version_for_mc(version_select.value)
        java_info_label.value = (
            f'<small style="color:#555">'
            f'☕ This version requires <b>Java {jv}</b> — installed automatically.'
            f'</small>'
        )

version_select.observe(_update_java_info, names='value')

def on_choice_change(change):
    """Reset UI state when the server selection changes."""
    versions_fetched['done'] = False
    versions_fetched['name'] = None
    with out:
        clear_output(wait=True)
    if change['new'] == '➕ Create a new server':
        version_select.options = []
        version_select.value   = None
        java_info_label.value  = ''
        name_box.layout.display    = 'flex'
        version_box.layout.display = 'none'
        setup_btn.description  = 'Fetch Versions'
        setup_btn.button_style = 'info'
        setup_btn.icon         = 'search'
    else:
        name_box.layout.display    = 'none'
        version_box.layout.display = 'none'
        setup_btn.description  = 'Load Server'
        setup_btn.button_style = 'success'
        setup_btn.icon         = 'play'

choice_select.observe(on_choice_change, names='value')
on_choice_change({'new': choice_select.value})

def on_setup_click(b):
    """Run _do_setup in a background thread so the UI stays responsive during downloads."""
    setup_btn.disabled = True
    def _run():
        try:
            _do_setup()
        finally:
            try:
                setup_btn.disabled = False
            except Exception:
                pass  # widget mutation from background thread
    threading.Thread(target=_run, daemon=True).start()

def _do_setup():
    with out:
        clear_output(wait=True)
        choice = choice_select.value

        # ── Load existing server ────────────────────────────
        if choice != '➕ Create a new server':
            server_path = SERVERS_DIR / choice

            # Check directory existence FIRST — avoids wasted metadata reads
            # and confusing errors if the folder was deleted.
            if not server_path.is_dir():
                print(f'❌ Server folder not found: {server_path}')
                print('   It may have been deleted. Try re-creating the server.')
                return

            meta_file = server_path / '.mc_meta.json'
            mc_ver   = None
            req_java = 17

            if meta_file.exists():
                try:
                    meta     = json.loads(meta_file.read_text(encoding='utf-8'))
                    mc_ver   = meta.get('mc_version')
                    req_java = meta.get('java') or java_version_for_mc(mc_ver or '1.20')
                    print(f'📋 Metadata: MC {mc_ver}, Java {req_java}')
                except Exception as e:
                    print(f'⚠️  Could not read metadata: {e}')

            jars = sorted(
                server_path.glob('paper-*.jar'),
                key=lambda p: p.stat().st_mtime, reverse=True
            )
            if not jars:
                jars = sorted(
                    server_path.glob('*.jar'),
                    key=lambda p: p.stat().st_mtime, reverse=True
                )
            if not jars:
                print(f'❌ No JAR file found in {server_path}.')
                print('   The folder may be corrupted. Try creating a new server.')
                return

            jar_path = jars[0]

            if mc_ver is None:
                try:
                    parts    = jar_path.stem.split('-')   # paper-1.21.1-123
                    mc_ver   = parts[1] if len(parts) >= 2 else '1.20'
                    req_java = java_version_for_mc(mc_ver)
                except Exception:
                    mc_ver, req_java = '1.20', 17

            java_bin = install_java(req_java)
            state['server_path'] = server_path
            state['server_name'] = choice
            state['jar_path']    = jar_path
            state['java_cmd']    = java_bin
            state['java_ver']    = req_java

            print(f'\n✅ Server "{choice}" is ready!')
            print(f'   MC version : {mc_ver}')
            print(f'   JAR        : {jar_path.name}')
            print(f'   Java       : Java {req_java}')
            print('\n▶ Run Cell 3 to start the server!')
            return

        # ── New server — step 1: validate name & fetch versions ─
        if not versions_fetched['done']:
            raw_name  = name_input.value.strip()
            if not raw_name:
                print('⚠️  Please enter a server name first!')
                return
            safe_name = ''.join(c for c in raw_name if c.isascii() and (c.isalnum() or c in '-_'))
            if not safe_name:
                print('⚠️  Server name must contain letters or numbers.')
                return
            if (SERVERS_DIR / safe_name).exists():
                print(f'⚠️  A server named "{safe_name}" already exists.')
                print('   Choose a different name, or select it from the list above.')
                return

            print('🔍 Fetching available PaperMC versions...')
            try:
                versions = get_paper_versions()
                if not versions:
                    print('❌ PaperMC returned no versions. Check your internet connection.')
                    return
                try:
                    version_select.options = versions
                    version_select.value   = versions[0]
                    version_box.layout.display = 'flex'
                except Exception:
                    pass  # widget state mutation from background thread may fail
                versions_fetched['done'] = True
                versions_fetched['name'] = safe_name
                try:
                    setup_btn.description  = 'Download & Set Up'
                    setup_btn.button_style = 'success'
                    setup_btn.icon         = 'download'
                except Exception:
                    pass
                try:
                    _update_java_info()
                except Exception:
                    pass
                print(f'✅ Found {len(versions)} versions.')
                print('   Select a version above, then click "Download & Set Up".')
            except Exception as e:
                print(f'❌ Failed to fetch versions: {e}')
            return

        # ── New server — step 2: download & configure ─────────
        mc_version = version_select.value
        if not mc_version:
            print('⚠️  No version selected.')
            return

        safe_name   = versions_fetched['name']
        if not safe_name:
            print('\u26a0\ufe0f  Server name was lost \u2014 please click \"Fetch Versions\" again.')
            return
        server_path = SERVERS_DIR / safe_name
        dir_created = not server_path.exists()
        server_path.mkdir(parents=True, exist_ok=True)

        print(f'🏗️  Creating server "{safe_name}" — Minecraft {mc_version}')
        print()

        setup_ok = False
        try:
            req_java = java_version_for_mc(mc_version)
            print(f'☕ Minecraft {mc_version} requires Java {req_java}')
            java_bin = install_java(req_java)
            print()

            build_info = get_latest_build(mc_version)
            if not build_info:
                print(f'❌ No builds available for Minecraft {mc_version}.')
                return

            try:
                jar_name = build_info['downloads']['application']['name']
            except (KeyError, TypeError) as e:
                print(f'\u274c Unexpected API response from PaperMC: {e}')
                return
            jar_path = server_path / jar_name

            # Skip download only if file is larger than 1 MB (guards corrupt partials)
            if jar_path.exists() and jar_path.stat().st_size > 1_000_000:
                expected_sha256 = build_info['downloads']['application'].get('sha256')
                if expected_sha256:
                    print('⚡ JAR cached — verifying checksum...')
                    ctx = hashlib.sha256()
                    with open(jar_path, 'rb') as _f:
                        while True:
                            _chunk = _f.read(65536)
                            if not _chunk:
                                break
                            ctx.update(_chunk)
                    if ctx.hexdigest() != expected_sha256:
                        print('⚠️  Cached JAR is corrupt — re-downloading...')
                        jar_path.unlink()
                        download_paper_jar(mc_version, build_info, jar_path, progress_bar)
                    else:
                        print('⚡ JAR already downloaded — checksum verified.')
                else:
                    print('⚡ JAR already downloaded — skipping.')
            else:
                if jar_path.exists():
                    jar_path.unlink()
                download_paper_jar(mc_version, build_info, jar_path, progress_bar)

            # Accept EULA — inform the user about what they are agreeing to
            print('📋 By continuing, you agree to the Minecraft EULA:')
            print('   https://www.minecraft.net/en-us/eula')
            (server_path / 'eula.txt').write_text('eula=true\n', encoding='utf-8')
            print('   EULA accepted.')

            # Write default server.properties only on first setup
            props_path = server_path / 'server.properties'
            if not props_path.exists():
                props_path.write_text(
                    '#Minecraft server properties\n'
                    'online-mode=false\n'
                    'server-port=25565\n'
                    'gamemode=survival\n'
                    'difficulty=normal\n'
                    'max-players=20\n'
                    'view-distance=10\n'
                    'simulation-distance=10\n'
                    'spawn-protection=0\n'
                    'enable-command-block=true\n'
                    f'motd={safe_name}\n'
                    'level-name=world\n'
                    'pvp=true\n'
                    'allow-flight=false\n'
                    'white-list=false\n',
                    encoding='utf-8'
                )
                print('⚙️  Created server.properties.')
                print('   ⚠️  WARNING: online-mode=false — any username can join.')
                print('   Edit this in Cell 4 to require purchased Minecraft accounts.')

            # Save metadata for reliable future loads
            # Atomic write — prevents corrupt metadata on unexpected interruption
            _meta_tmp = server_path / '.mc_meta.json.tmp'
            _meta_content = json.dumps(
                {'name': safe_name, 'mc_version': mc_version,
                 'jar': jar_name, 'java': req_java}, indent=2
            )
            try:
                _meta_tmp.write_text(_meta_content, encoding='utf-8')
                _meta_tmp.replace(server_path / '.mc_meta.json')
            except Exception:
                if _meta_tmp.exists():
                    _meta_tmp.unlink()
                raise

            state['server_path'] = server_path
            state['server_name'] = safe_name
            state['jar_path']    = jar_path
            state['java_cmd']    = java_bin
            state['java_ver']    = req_java

            print(f'\n🎉 Server "{safe_name}" is ready!')
            print(f'   Path : {server_path}')
            print(f'   JAR  : {jar_name}')
            print(f'   Java : Java {req_java}')
            print('\n▶ Run Cell 3 to start the server!')
            setup_ok = True

            # Refresh the server list so the new server appears without re-running Cell 2
            try:
                _updated = sorted(
                    d.name for d in SERVERS_DIR.iterdir()
                    if d.is_dir() and not d.name.startswith('.')
                )
                choice_select.options = ['➕ Create a new server'] + _updated
                choice_select.value   = safe_name
            except Exception:
                pass  # widget mutation from background thread may fail silently

        finally:
            # Clean up on failure — only remove empty dirs to avoid losing downloaded JARs
            if not setup_ok and dir_created and server_path.exists():
                try:
                    contents = list(server_path.iterdir())
                    if not contents:
                        server_path.rmdir()
                        print('🧹 Cleaned up empty server directory.')
                    # Do NOT rmtree — preserve any downloaded JAR for retry
                except Exception:
                    pass

setup_btn.on_click(on_setup_click)

display(widgets.VBox([
    widgets.HTML('<h3 style="margin:4px 0">🗂️ Server Selection</h3>'),
    choice_select,
    name_box,
    version_box,
    progress_bar,
    setup_btn,
    out,
]))


In [ ]:
# ============================================================
# CELL 3 — Start Server & Console
# ============================================================

if ('state' not in globals()
        or not state.get('server_path')
        or not state.get('jar_path')
        or not state.get('java_cmd')
        or not state.get('server_name')
        or 'build_jvm_flags' not in globals()
        or '_html_mod' not in globals()):
    print('❌ No server configured!')
    print('   Please run Cell 1 and Cell 2 first, and make sure setup completed successfully.')
else:
    # Kill any previously running server so port 25565 is free
    _prev_proc = globals().get('_mc_proc')
    if _prev_proc is not None and _prev_proc.poll() is None:
        print('⚠️  Stopping previous server first...')
        try:
            if _prev_proc.stdin and not _prev_proc.stdin.closed:
                _prev_proc.stdin.write('stop\n')
                _prev_proc.stdin.flush()
            _prev_proc.wait(timeout=15)
        except Exception:
            try:
                _prev_proc.kill()
                _prev_proc.wait(timeout=3)
            except Exception:
                pass
        print('   Previous server stopped.')
        time.sleep(2)

    server_path = state['server_path']
    jar_path    = state['jar_path']
    java_cmd    = state['java_cmd']
    server_name = state['server_name']
    java_ver    = state.get('java_ver') or 17

    jvm_flags = build_jvm_flags(java_ver)

    print(f'🚀 Starting "{server_name}"...')
    print(f'   JAR  : {jar_path.name}')
    print(f'   Java : Java {java_ver}  ({java_cmd})')
    _xms, _xmx = _detect_ram()
    print(f'   RAM  : Xms{_xms}M / Xmx{_xmx}M  (auto-detected from system memory)')
    print()

    _mc_proc = subprocess.Popen(
        [java_cmd] + jvm_flags + ['-jar', str(jar_path), '--nogui'],
        cwd=str(server_path),
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',   # never crash on invalid UTF-8 from the server
        bufsize=1,
    )

    # Register atexit handler so the server is stopped if the kernel exits.
    # Uses a closure over _mc_proc to avoid affecting future Cell 3 re-runs.
    def _atexit_kill(_proc=_mc_proc):
        if _proc.poll() is None:
            try:
                if _proc.stdin and not _proc.stdin.closed:
                    _proc.stdin.write('stop\n')
                    _proc.stdin.flush()
                _proc.wait(timeout=10)
            except Exception:
                try:
                    _proc.kill()
                except Exception:
                    pass
    # Unregister any previous atexit handler to avoid accumulation on Cell 3 re-runs
    _prev_atexit = globals().get('_mc_atexit_handler')
    if _prev_atexit is not None:
        try:
            atexit.unregister(_prev_atexit)
        except Exception:
            pass
    _mc_atexit_handler = _atexit_kill
    atexit.register(_atexit_kill)

    # Thread-safe lock for log_lines — protects against concurrent mutation
    # from the reader thread, update loop, and UI callbacks.
    _log_lock = threading.Lock()
    log_lines = []
    log_queue = queue.Queue()
    _stop_requested = {'value': False}   # mutable flag for stop-button retry logic
    _stdin_lock = threading.Lock()        # protects _mc_proc.stdin writes

    # Strip Minecraft ANSI/VT100 color codes so they don't appear as garbage
    _ANSI_RE = re.compile(r'\x1b(?:[@-Z\\-_]|\[[0-9;?]*[ -/]*[@-~])')

    def _reader(_stdout=_mc_proc.stdout):
        """Read server stdout line-by-line; strip ANSI codes; push to queue."""
        try:
            for raw_line in _stdout:
                clean = _ANSI_RE.sub('', raw_line.rstrip())
                log_queue.put(clean)
        except Exception as _e:
            log_queue.put(f'[reader error] {_e}')
        finally:
            log_queue.put(None)   # sentinel: stdout closed

    threading.Thread(target=_reader, daemon=True).start()

    # ── Console widgets ───────────────────────────────────
    console_area = widgets.Textarea(
        value='',
        placeholder='Server output will appear here...',
        layout=widgets.Layout(width='100%', height='420px'),
        disabled=True,
    )
    console_area.add_class('mc-console')

    cmd_input = widgets.Text(
        placeholder='Type a command and press Enter (e.g.  list  |  say Hi  |  stop)',
        layout=widgets.Layout(flex='1'),
    )
    send_btn = widgets.Button(
        description='Send',
        button_style='primary',
        icon='terminal',
        layout=widgets.Layout(width='100px'),
    )
    stop_btn = widgets.Button(
        description='Stop Server',
        button_style='danger',
        icon='stop',
        layout=widgets.Layout(width='130px'),
    )
    status_label = widgets.HTML('<b style="color:orange">⏳ Starting...</b>')

    MAX_LINES = 500

    # Escape server_name for safe HTML rendering (prevents XSS from dir names)
    _safe_name_html = _html_mod.escape(server_name)

    def _refresh():
        with _log_lock:
            console_area.value = '\n'.join(log_lines[-MAX_LINES:])

    def _ingest_line(line):
        """Append a single log line, trim buffer, and detect server-online status."""
        with _log_lock:
            log_lines.append(line)
            if len(log_lines) > MAX_LINES * 4:
                del log_lines[:MAX_LINES * 2]
        if 'Done' in line and 'For help' in line:
            status_label.value = '<b style="color:green">🟢 Server is ONLINE</b>'

    def _update_loop(_proc=_mc_proc):
        """Background thread: drain queue and push updates to the console widget."""
        while True:
            changed = False
            done    = False

            while True:
                try:
                    item = log_queue.get_nowait()
                except queue.Empty:
                    break

                if item is None:   # sentinel — drain any remaining items first
                    while True:
                        try:
                            leftover = log_queue.get_nowait()
                            if leftover is not None:
                                _ingest_line(leftover)
                                changed = True
                        except queue.Empty:
                            break
                    done = True
                    break

                _ingest_line(item)
                changed = True

            if changed:
                try:
                    _refresh()
                except Exception:
                    pass  # widget may have been garbage-collected; keep thread alive

            if done:
                try:
                    _refresh()
                except Exception:
                    pass
                status_label.value = '<b style="color:red">🔴 Server stopped</b>'
                send_btn.disabled  = True
                cmd_input.disabled = True
                stop_btn.disabled  = True
                try:
                    _proc.wait(timeout=5)
                except Exception:
                    pass
                return

            time.sleep(0.2)

    threading.Thread(target=_update_loop, daemon=True).start()

    MAX_CMD_LEN = 1024

    def send_command(b=None):
        """Send the typed command to the server stdin."""
        cmd = cmd_input.value.strip()
        if not cmd:
            return
        if len(cmd) > MAX_CMD_LEN:
            with _log_lock:
                log_lines.append(f'[console] Command too long ({len(cmd)} chars, max {MAX_CMD_LEN}).')
            _refresh()
            cmd_input.value = ''
            return
        if _mc_proc.poll() is not None:
            with _log_lock:
                log_lines.append('[console] Server is not running.')
            _refresh()
            cmd_input.value = ''
            return
        try:
            with _stdin_lock:
                _mc_proc.stdin.write(cmd + '\n')
                _mc_proc.stdin.flush()
            with _log_lock:
                log_lines.append(f'> {cmd}')
            _refresh()
        except Exception as e:
            with _log_lock:
                log_lines.append(f'[console error] {e}')
            _refresh()
        cmd_input.value = ''

    def stop_server(b):
        """Send save-all then stop. Force-kills on second click if server ignores the command."""
        stop_btn.disabled = True

        # Second click after reactivate — force-kill immediately
        if _stop_requested['value']:
            with _log_lock:
                log_lines.append('[console] 🔪 Force-killing server (SIGKILL)...')
            _refresh()
            try:
                _mc_proc.kill()
            except Exception:
                pass
            return

        _stop_requested['value'] = True
        with _log_lock:
            log_lines.append('[console] Saving world and stopping server...')
        _refresh()
        try:
            with _stdin_lock:
                # save-all then stop — server processes commands in order, no sleep needed
                _mc_proc.stdin.write('save-all\n')
                _mc_proc.stdin.flush()
                _mc_proc.stdin.write('stop\n')
                _mc_proc.stdin.flush()
        except Exception as _e:
            with _log_lock:
                log_lines.append(f'[console] ⚠️  Failed to send stop command ({_e}) — force-killing server...')
            _refresh()
            try:
                _mc_proc.terminate()
            except Exception:
                pass
            def _force_kill(_proc=_mc_proc, _lock=_log_lock, _lines=log_lines, _rfn=_refresh):
                time.sleep(8)
                if _proc.poll() is None:
                    try:
                        _proc.kill()
                    except Exception:
                        pass
                    with _lock:
                        _lines.append('[console] 🔪 Server force-killed (SIGKILL).')
                    _rfn()
                # If process already exited from terminate(), nothing more to do.
            threading.Thread(target=_force_kill, daemon=True).start()
            return

        # Capture current objects by default-arg so a Cell 3 re-run within 20s
        # doesn't cause _reactivate to affect the new run's widgets/process.
        def _reactivate(
            _proc=_mc_proc, _btn=stop_btn,
            _lock=_log_lock, _lines=log_lines, _rfn=_refresh
        ):
            time.sleep(20)
            if _proc.poll() is None:   # still running after 20 s
                _btn.disabled = False
                with _lock:
                    _lines.append(
                        '[console] ⚠️  Server has not stopped yet — '
                        'click Stop again to force-kill it.'
                    )
                _rfn()

        threading.Thread(target=_reactivate, daemon=True).start()

    send_btn.on_click(send_command)
    cmd_input.on_submit(send_command)
    stop_btn.on_click(stop_server)

    # ── Quick-action buttons ──────────────────────────────
    def _quick(label, cmd_str):
        b = widgets.Button(
            description=label,
            layout=widgets.Layout(width='auto', margin='2px'),
        )
        def _click(x):
            cmd_input.value = cmd_str
            send_command()
        b.on_click(_click)
        return b

    quick_btns = widgets.HBox([
        _quick('📋 list players',  'list'),
        _quick('🕐 set day',       'time set day'),
        _quick('☀️ clear weather', 'weather clear'),
        _quick('💾 save-all',      'save-all'),
        _quick('📢 say Hello',     'say Hello!'),
        _quick('🎮 creative',      'gamemode creative @a'),
        _quick('⚔️  survival',     'gamemode survival @a'),
        _quick('🗺️  seed',         'seed'),
    ])

    header = widgets.HBox(
        [
            widgets.HTML(f'<h3 style="margin:0">🎮 {_safe_name_html}</h3>'),
            widgets.HTML('&nbsp;|&nbsp;'),
            status_label,
            widgets.HTML('&nbsp;&nbsp;'),
            stop_btn,
        ],
        layout=widgets.Layout(align_items='center'),
    )

    display(widgets.VBox([
        header,
        widgets.HTML('<hr style="margin:4px 0">'),
        console_area,
        widgets.HTML('<hr style="margin:4px 0"><b>⚡ Quick Actions:</b>'),
        quick_btns,
        widgets.HTML('<hr style="margin:4px 0"><b>📟 Send Command:</b>'),
        widgets.HBox([cmd_input, send_btn]),
        widgets.HTML(
            '<small style="color:#888">'
            f'💾 World saved in MinecraftServers/{_safe_name_html}/world/'
            '</small>'
        ),
    ]))


In [ ]:
# ============================================================
# CELL 4 (Optional) — View & Edit Server Settings
# ============================================================
# Edit server.properties directly in the browser.
# Changes take effect the next time the server starts.
# ============================================================

if ('state' not in globals() or not state.get('server_path')
        or '_html_mod' not in globals()):
    print('❌ No server loaded. Run Cell 1 and Cell 2 first.')
else:
    props_path = state['server_path'] / 'server.properties'

    if not props_path.exists():
        print('⚠️  server.properties not found.')
        print('   Start the server at least once (Cell 3) to generate it, then re-run this cell.')
    else:
        editor = widgets.Textarea(
            value=props_path.read_text(encoding='utf-8'),
            layout=widgets.Layout(width='100%', height='420px'),
        )
        editor.add_class('mc-editor')   # monospace CSS from Cell 1

        save_btn   = widgets.Button(
            description='💾 Save Changes',
            button_style='success',
            layout={'width': '180px', 'margin': '8px 4px 8px 0'},
        )
        reload_btn = widgets.Button(
            description='🔄 Reload File',
            button_style='',
            layout={'width': '150px', 'margin': '8px 0'},
        )

        msg_out = widgets.Output()

        def on_save(b):
            """Save the editor content to server.properties via atomic write."""
            save_btn.disabled = True
            try:
                content = editor.value
                # Guard against accidentally saving an empty or near-empty file
                if len(content.strip()) < 20:
                    with msg_out:
                        clear_output(wait=True)
                        print('⚠️  Content is too short to be a valid server.properties — not saved.')
                        print('   Click "🔄 Reload File" to restore the original.')
                    return
                tmp_path = props_path.parent / (props_path.name + '.tmp')
                try:
                    tmp_path.write_text(content, encoding='utf-8')
                    tmp_path.replace(props_path)
                except Exception:
                    if tmp_path.exists():
                        tmp_path.unlink()
                    raise
                with msg_out:
                    clear_output(wait=True)
                    print('✅ Saved! Restart the server (re-run Cell 3) for changes to take effect.')
            except Exception as e:
                with msg_out:
                    clear_output(wait=True)
                    print(f'❌ Save failed: {e}')
            finally:
                save_btn.disabled = False

        def on_reload(b):
            """Reload server.properties from disk into the editor."""
            try:
                editor.value = props_path.read_text(encoding='utf-8')
                with msg_out:
                    clear_output(wait=True)
                    print('🔄 Reloaded from disk.')
            except Exception as e:
                with msg_out:
                    clear_output(wait=True)
                    print(f'❌ Reload failed: {e}')

        save_btn.on_click(on_save)
        reload_btn.on_click(on_reload)

        _safe_props_html = _html_mod.escape(str(props_path))
        display(widgets.VBox([
            widgets.HTML(
                '<h3 style="margin:4px 0">⚙️ server.properties</h3>'
                f'<small style="color:#888">{_safe_props_html}</small>'
            ),
            editor,
            widgets.HBox([save_btn, reload_btn]),
            msg_out,
            widgets.HTML(
                '<small style="color:#888">'
                'Key settings: '
                '<b>gamemode</b> (survival / creative / adventure) · '
                '<b>difficulty</b> (peaceful / easy / normal / hard) · '
                '<b>max-players</b> · '
                '<b>pvp</b> (true / false) · '
                '<b>online-mode</b> (true = premium accounts only)'
                '</small>'
            ),
        ]))


---

## 📋 Tips & FAQ

| Question | Answer |
|---|---|
| **Where are my worlds?** | `MinecraftServers/<YourServerName>/world/` (next to this notebook) |
| **How do I change RAM?** | RAM is auto-detected (75% of system memory, max 8 GB). Override by editing `_detect_ram()` in Cell 2 |
| **How do I install plugins?** | Drop `.jar` files into `MinecraftServers/<name>/plugins/`, then restart |
| **How do I connect from Minecraft?** | Use a playit.gg or ngrok TCP tunnel pointing at port 25565 |
| **How do I update PaperMC?** | Delete the old `.jar` from `MinecraftServers/<name>/`, then re-run Cell 2 |
| **online-mode=false — safe?** | Lets non-premium accounts join. Edit it in Cell 4 to require purchased accounts |
| **Can I run multiple servers?** | Yes — create each with a different name and a different `server-port` in Cell 4 |
| **Java version error?** | Java is installed automatically via apt-get (Colab/Linux) or Nix (Replit) |
| **Server starts then crashes?** | Check the console for errors. Common causes: out of RAM, corrupt world |
| **Save-all before stopping?** | The Stop button sends `save-all` automatically before sending `stop` |
